In [10]:
import pandas as pd
import numpy as np
import torch
import duckdb

In [3]:

asa_selection = pd.read_csv("/home/gagan/open_audio_datasets/asa_berlin/ASA_Berlin_Cornell_Challenge/ASA_Berlin_Selection.csv")

In [4]:
asa_selection.head(5)

,gbifID,species,countryCode,locality,decimalLatitude,decimalLongitude,eventDate,day,month,year,catalogNumber,recordedBy,filename,ebird_code
0,1500204467,Anas acuta,DE,Dingden,51.778889,6.677222,2017-03-26T20:15:00,26.0,3.0,2017.0,TSA:Anas_acuta_DIG_195_3_0,"Wesseler, Thomas",Anas_acuta_DIG0195_03_short,norpin
1,1500204466,Anas acuta,DE,Dingden,51.778889,6.677222,2017-03-26T20:15:00,26.0,3.0,2017.0,TSA:Anas_acuta_DIG_195_5_0,"Wesseler, Thomas",Anas_acuta_DIG0195_05_short,norpin
2,1500204465,Anas acuta,DE,Dingden,51.778889,6.677222,2017-03-26T20:15:00,26.0,3.0,2017.0,TSA:Anas_acuta_DIG_195_1_0,"Wesseler, Thomas",Anas_acuta_DIG0195_01_short,norpin
3,1500204464,Anas acuta,DE,Dingden,51.778889,6.677222,2017-03-26T20:15:00,26.0,3.0,2017.0,TSA:Anas_acuta_DIG_195_4_0,"Wesseler, Thomas",Anas_acuta_DIG0195_04_short,norpin
4,1500204463,Anas acuta,DE,Dingden,51.778889,6.677222,2017-03-26T20:15:00,26.0,3.0,2017.0,TSA:Anas_acuta_DIG_195_2_0,"Wesseler, Thomas",Anas_acuta_DIG0195_02_short,norpin


In [5]:
asa_selection.shape

(557, 14)

In [13]:
asa_selection.species.unique()

array(['Anas acuta', 'Plectrophenax nivalis', 'Passer domesticus',
       'Pandion haliaetus', 'Corvus corax', 'Clangula hyemalis',
       'Acanthis flammea', 'Vireo olivaceus', 'Turdus migratorius',
       'Sturnus vulgaris', 'Riparia riparia', 'Passerina caerulea',
       'Setophaga striata', 'Hirundo rustica', 'Cardinalis cardinalis',
       'Oxyura jamaicensis', 'Meleagris gallopavo',
       'Lophodytes cucullatus', 'Cygnus columbianus', 'Cygnus buccinator',
       'Cyanocitta stelleri', 'Callipepla californica',
       'Buteo jamaicensis', 'Branta canadensis', 'Ardea alba',
       'Aquila chrysaetos', 'Anas platyrhynchos', 'Alectoris chukar',
       'Aix sponsa', 'Haliaeetus leucocephalus', 'Bubo virginianus',
       'Sterna hirundo', 'Phalaropus lobatus', 'Larus argentatus',
       'Columba livia', 'Troglodytes aedon', 'Loxia curvirostra',
       'Vireo cassinii', 'Asio flammeus', 'Zonotrichia leucophrys',
       'Podiceps nigricollis', 'Cardellina pusilla', 'Anthus rubescens',
 

In [12]:
conn = duckdb.connect("/home/gagan/esp_projects/taxonomy/app/gbif_taxonomy.db")

In [11]:
def get_taxonomy_and_update(sci_name: str) -> dict:
    """
    Get GBIF taxonomy data for a given scientific name and update the database if necessary.
    """
    # Use parameterized query
    query = """SELECT TaxonID as gbifID, Kingdom as kingdom, Phylum as phylum, 
               "Class" as class, "Order" as order, Family as family,
               Genus as genus, CommonNames as species_common
               FROM gbif WHERE ScientificName = ?"""
    
    print(f"Checking database for: {sci_name}")
    res = conn.execute(query, [sci_name]).fetch_df()
    print(f"Query result shape: {res.shape}")
    
    if res.empty:
        print(f"Looking for {sci_name} in gbif API") 
        data = species.name_backbone(sci_name)
        time.sleep(1.5)
        
        if data and data["status"] == "ACCEPTED":
            print(f"Found it!, updating db")
            if data.get("usageKey"):
                try:
                    # Use parameterized insert
                    update_query = """INSERT INTO gbif (TaxonID, ScientificName, Kingdom, Phylum, "Class", "Order", Family, Genus, CommonNames)
                        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)"""
                    
                    conn.execute(update_query, [
                        data["usageKey"],
                        data.get("scientificName", sci_name),
                        data.get("kingdom", ""),
                        data.get("phylum", ""),
                        data.get("class", ""),
                        data.get("order", ""),
                        data.get("family", ""),
                        data.get("genus", ""),
                        data.get("commonName", [""])[0] if data.get("commonName") else ""
                    ])
                    conn.commit()
                    
                    # Verify the insert worked
                    verify_res = conn.execute(query, [sci_name]).fetch_df()
                    print(f"After insert, query result shape: {verify_res.shape}")
                    
                except Exception as e:
                    print(f"Error inserting data: {e}")

                return {
                    "gbifID": data["usageKey"],
                    "kingdom": data.get("kingdom", ""),
                    "phylum": data.get("phylum", ""),
                    "class": data.get("class", ""),
                    "order": data.get("order", ""),
                    "family": data.get("family", ""),
                    "genus": data.get("genus", ""),
                    "species_common": data.get("commonName", [""])[0] if data.get("commonName") else "",
                }
        
        # Return empty result if not found
        return {
            "gbifID": None,
            "kingdom": "",
            "phylum": "",
            "class": "",
            "order": "",
            "family": "",
            "genus": "",
            "species_common": "",
        }
    else:
        print(f"Found in database!")
        return res.iloc[0].to_dict()

In [14]:
conn.close()